# HaxBall PPO local: painel interativo

Este notebook controla o treinador local em `train_haxball_rl_local.py`. Os controles abaixo alteram os hiperparâmetros sem editar código. O treino roda em segundo plano, salva checkpoints e atualiza os gráficos no final de cada execução.

- `Iniciar / retomar`: começa do zero ou continua do checkpoint informado.
- `Parar com segurança`: solicita parada entre iterações e preserva `haxball_rl_latest.pt`.
- `Atualizar gráficos`: lê `training_metrics.json` e `training_dashboard.png`.

Execute as células em ordem. O notebook deve ser aberto a partir deste repositório.

In [1]:
from pathlib import Path
import json
import threading
import traceback
from IPython.display import clear_output, display, Image
import ipywidgets as widgets
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(path for path in candidates if (path / 'train_haxball_rl_local.py').exists())
if str(ROOT) not in __import__('sys').path:
    __import__('sys').path.insert(0, str(ROOT))
import train_haxball_rl_local as trainer

print(f'Repositório: {ROOT}')
print('Backend local carregado.')

Repositório: c:\Users\caihe\Documents\antigravity\agitated-hertz
Backend local carregado.


In [7]:
# Controles principais
mapa = widgets.Dropdown(options=sorted(path.stem for path in (ROOT / 'haxball' / 'maps').glob('*.hbs')), value='futsal_2v2', description='Mapa:', layout=widgets.Layout(width='360px'))
iteracoes = widgets.IntSlider(value=100, min=1, max=2000, step=1, description='Iterações:', continuous_update=False, layout=widgets.Layout(width='420px'))
passos = widgets.IntSlider(value=2048, min=128, max=8192, step=128, description='Passos:', continuous_update=False, layout=widgets.Layout(width='420px'))
batch = widgets.IntSlider(value=128, min=16, max=512, step=16, description='Batch:', continuous_update=False, layout=widgets.Layout(width='420px'))
epochs = widgets.IntSlider(value=8, min=1, max=20, step=1, description='Épocas:', continuous_update=False, layout=widgets.Layout(width='420px'))
eval_freq = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Eval a cada:', continuous_update=False, layout=widgets.Layout(width='420px'))
eval_matches = widgets.IntSlider(value=2, min=1, max=20, step=1, description='Partidas eval:', continuous_update=False, layout=widgets.Layout(width='420px'))
max_steps = widgets.IntSlider(value=1500, min=300, max=5000, step=100, description='Passos/ep.:', continuous_update=False, layout=widgets.Layout(width='420px'))
learning_rate = widgets.FloatLogSlider(value=3e-4, base=10, min=-5, max=-2, step=0.1, description='Learning rate:', readout_format='.1e', continuous_update=False, layout=widgets.Layout(width='420px'))
entropy = widgets.FloatSlider(value=0.012, min=0.0, max=0.05, step=0.001, description='Entropia:', readout_format='.3f', continuous_update=False, layout=widgets.Layout(width='420px'))
seed = widgets.IntText(value=42, description='Seed:', layout=widgets.Layout(width='200px'))
device = widgets.Dropdown(options=['auto', 'cpu', 'cuda'], value='auto', description='Device:', layout=widgets.Layout(width='240px'))
plot_mode = widgets.Dropdown(options=[('Salvar PNG', 'save'), ('Painel ao vivo', 'live'), ('Desligado', 'off')], value='save', description='Gráficos:', layout=widgets.Layout(width='300px'))
checkpoint = widgets.Text(value='', description='Warm start:', placeholder='checkpoints/haxball_rl_latest.pt', layout=widgets.Layout(width='620px'))
diretorio = widgets.Text(value='checkpoints/interactive_local', description='Pasta:', layout=widgets.Layout(width='620px'))

preset_smoke = widgets.Button(description='Preset teste rápido', button_style='info', icon='flask')
preset_train = widgets.Button(description='Preset treino longo', button_style='primary', icon='rocket')
start = widgets.Button(description='Iniciar / retomar', button_style='success', icon='play')
stop = widgets.Button(description='Parar com segurança', button_style='warning', icon='stop')
refresh = widgets.Button(description='Atualizar gráficos', icon='refresh')
save = widgets.Button(description='Mostrar checkpoints', icon='save')
status = widgets.HTML('<b>Status:</b> pronto')
output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))

controls = widgets.VBox([
    widgets.HTML('<h3>Configuração do treino</h3>'),
    widgets.HBox([mapa, device, plot_mode]),
    widgets.HBox([iteracoes, passos]),
    widgets.HBox([batch, epochs]),
    widgets.HBox([eval_freq, eval_matches]),
    widgets.HBox([max_steps, seed]),
    widgets.HBox([learning_rate, entropy]),
    checkpoint, diretorio,
    widgets.HBox([preset_smoke, preset_train]),
    widgets.HBox([start, stop, refresh, save]),
    status, output,
])

In [8]:
# Ações e visualização
run_state = {'thread': None, 'stop_event': None}

def set_status(text, color='#444'):
    status.value = f"<b>Status:</b> <span style='color:{color}'>{text}</span>"

def config_from_widgets(stop_event=None):
    warm = checkpoint.value.strip()
    warm_path = (ROOT / warm).resolve() if warm else None
    if warm_path and not warm_path.exists():
        raise FileNotFoundError(f'Warm start não encontrado: {warm_path}')
    save_dir = (ROOT / diretorio.value.strip()).resolve()
    return trainer.Config(
        map_name=mapa.value, iterations=iteracoes.value, steps=passos.value,
        batch_size=batch.value, update_epochs=epochs.value,
        eval_frequency=eval_freq.value, eval_matches=eval_matches.value,
        max_steps=max_steps.value, learning_rate=learning_rate.value,
        entropy_coef=entropy.value, seed=seed.value, device=device.value,
        plot_mode=plot_mode.value, checkpoint_dir=save_dir,
        warm_start=warm_path, stop_event=stop_event,
    )

def show_metrics():
    folder = (ROOT / diretorio.value.strip()).resolve()
    metrics_path = folder / 'training_metrics.json'
    image_path = folder / 'training_dashboard.png'
    with output:
        clear_output(wait=True)
        if metrics_path.exists():
            data = json.loads(metrics_path.read_text(encoding='utf-8'))
            if data:
                last = data[-1]
                print(f"Iteração {last['iteration']} | recompensa {last['mean_reward']:+.3f} | win rate {last['train_win_rate']:.1f}%")
        if image_path.exists():
            display(Image(filename=str(image_path)))
        else:
            print('Ainda não existe um dashboard salvo.')

def start_training(_=None):
    if run_state['thread'] is not None and run_state['thread'].is_alive():
        set_status('treino já está rodando', '#c47a00')
        return
    try:
        stop_event = threading.Event()
        config = config_from_widgets(stop_event)
    except Exception as error:
        set_status(str(error), '#b3261e')
        return

    def worker():
        set_status('treinando em segundo plano...', '#1769aa')
        try:
            trainer.train(config)
            set_status('treino concluído e salvo', '#188038')
        except Exception:
            set_status('erro no treino; veja a saída abaixo', '#b3261e')
            with output:
                traceback.print_exc()
        finally:
            show_metrics()

    run_state['stop_event'] = stop_event
    run_state['thread'] = threading.Thread(target=worker, daemon=True)
    run_state['thread'].start()

def stop_training(_=None):
    event = run_state.get('stop_event')
    if event is None:
        set_status('nenhum treino ativo', '#666')
    else:
        event.set()
        set_status('parada solicitada; aguardando fim da iteração', '#c47a00')

def apply_smoke(_=None):
    iteracoes.value, passos.value, batch.value, epochs.value = 3, 512, 64, 2
    eval_freq.value, eval_matches.value, max_steps.value = 1, 1, 600
    plot_mode.value = 'save'
    set_status('preset de teste rápido aplicado', '#1769aa')

def apply_train(_=None):
    iteracoes.value, passos.value, batch.value, epochs.value = 100, 2048, 128, 8
    eval_freq.value, eval_matches.value, max_steps.value = 10, 2, 1500
    plot_mode.value = 'save'
    set_status('preset de treino longo aplicado', '#1769aa')

def show_checkpoints(_=None):
    folder = (ROOT / diretorio.value.strip()).resolve()
    with output:
        clear_output(wait=True)
        for path in sorted(folder.glob('*.pt')):
            print(f'{path.name}: {path.stat().st_size / 1024:.1f} KB')

start.on_click(start_training)
stop.on_click(stop_training)
refresh.on_click(lambda _: show_metrics())
save.on_click(show_checkpoints)
preset_smoke.on_click(apply_smoke)
preset_train.on_click(apply_train)
set_status('controles prontos')

display(controls)
show_metrics()

## Fluxo recomendado

1. Clique em **Preset teste rápido** e depois em **Iniciar / retomar**.
2. Confira o PNG e as métricas com **Atualizar gráficos**.
3. Para um treino maior, clique em **Preset treino longo**.
4. Para continuar depois, informe `checkpoints/interactive_local/haxball_rl_latest.pt` no campo **Warm start**.
5. Use **Parar com segurança**; o último checkpoint concluído será preservado.

## Benchmark pós-treino: evolução das gerações

Esta seção compara automaticamente todos os `gen_iter_*.pt`, `haxball_rl_best.pt` e `haxball_rl_final.pt` encontrados na pasta de checkpoints. Ela mede desempenho real em partidas 1v1 contra os arquétipos, sem usar a recompensa interna do treino.

Métricas: taxa de vitória, empates, gols pró/contra, saldo, chutes, toques na bola e distância média até a bola. O relatório também cria gráficos de evolução e gera insights sobre progresso, estabilidade e especialização por adversário.

In [ ]:
# Controles do benchmark pós-treino
from pathlib import Path
import re
import threading
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from haxball.bots import BankBot, CounterBot, DribblerBot, GoalieBot, MasterBot, PressBot, RLBot, StrikerBot, WallReboundBot, HeuristicBot
from haxball.core.constants import Team
from haxball.core.game import HaxBallGame
from haxball.core.stadium import Stadium

benchmark_matches = widgets.IntSlider(value=3, min=1, max=20, step=1, description='Partidas:', continuous_update=False)
benchmark_frames = widgets.IntSlider(value=2400, min=600, max=6000, step=300, description='Frames:', continuous_update=False)
benchmark_stride = widgets.IntSlider(value=1, min=1, max=10, step=1, description='A cada N gerações:', continuous_update=False)
benchmark_mode = widgets.Dropdown(options=[('Todas as gerações', 'all'), ('Últimas 5', 'last5'), ('Marcos + final', 'milestones')], value='all', description='Modelos:')
benchmark_run = widgets.Button(description='Rodar benchmark', button_style='success', icon='bar-chart')
benchmark_stop = widgets.Button(description='Parar benchmark', button_style='warning', icon='stop')
benchmark_output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
benchmark_status = widgets.HTML('<b>Benchmark:</b> pronto')

display(widgets.VBox([
    widgets.HBox([benchmark_matches, benchmark_frames]),
    widgets.HBox([benchmark_stride, benchmark_mode]),
    widgets.HBox([benchmark_run, benchmark_stop]),
    benchmark_status,
    benchmark_output,
]))

In [10]:
# Motor do benchmark e geração de insights
benchmark_state = {'thread': None, 'stop': None}

BOT_FACTORIES_BENCHMARK = {
    'Heuristic': HeuristicBot,
    'Striker': StrikerBot,
    'Wall': WallReboundBot,
    'Press': PressBot,
    'Dribbler': DribblerBot,
    'Bank': BankBot,
    'Counter': CounterBot,
    'Master': MasterBot,
    'Goalie': GoalieBot,
}


def list_generation_checkpoints(folder, mode='all', stride=1):
    paths = list(folder.glob('gen_iter_*.pt'))
    paths += [folder / name for name in ('gen_stage1.pt', 'haxball_rl_best.pt', 'haxball_rl_final.pt') if (folder / name).exists()]
    unique = {path.resolve(): path for path in paths}

    def step_of(path):
        match = re.search(r'gen_iter_(\d+)', path.name)
        if match:
            return int(match.group(1))
        if 'stage1' in path.name:
            return 0
        if 'best' in path.name:
            return 10**9 - 2
        return 10**9 - 1

    ordered = sorted(unique.values(), key=lambda path: (step_of(path), path.name))
    generation_paths = [path for path in ordered if path.name.startswith('gen_iter_')]
    if mode == 'last5':
        selected = generation_paths[-5:]
    elif mode == 'milestones':
        selected = [path for path in ordered if path.name in {'gen_stage1.pt', 'gen_iter_250.pt', 'gen_iter_500.pt', 'gen_iter_1000.pt', 'gen_iter_1500.pt', 'gen_iter_2000.pt', 'haxball_rl_best.pt', 'haxball_rl_final.pt'}]
    else:
        selected = generation_paths[::max(1, stride)]
        selected += [path for path in ordered if path.name in {'haxball_rl_best.pt', 'haxball_rl_final.pt'}]
    return sorted({path.resolve(): path for path in selected}.values(), key=lambda path: (step_of(path), path.name))


def run_benchmark_match(red_bot, blue_bot, stadium, frames, stop_event):
    game = HaxBallGame(stadium=stadium, score_limit=3, time_limit_secs=0)
    red_bot.reset()
    blue_bot.reset()
    red = next(player for player in game.players if player.team == Team.RED)
    blue = next(player for player in game.players if player.team == Team.BLUE)
    red_kicks = blue_kicks = red_touches = blue_touches = 0
    red_distance = blue_distance = 0.0

    for _ in range(frames):
        if stop_event.is_set():
            break
        red_cmd = red_bot.act(game, red)
        blue_cmd = blue_bot.act(game, blue)
        info = game.step({red.player_id: red_cmd, blue.player_id: blue_cmd})
        red_kicks += int(red_cmd[2])
        blue_kicks += int(blue_cmd[2])
        for collision in info.get('events', {}).get('disc_ball_collisions', []):
            player_id = collision.get('player_id')
            if player_id == red.player_id:
                red_touches += 1
            elif player_id == blue.player_id:
                blue_touches += 1
        red_distance += red.pos.distance_to(game.ball.pos)
        blue_distance += blue.pos.distance_to(game.ball.pos)

    return {
        'red_score': game.red_score,
        'blue_score': game.blue_score,
        'red_kicks': red_kicks,
        'blue_kicks': blue_kicks,
        'red_touches': red_touches,
        'blue_touches': blue_touches,
        'red_distance': red_distance / max(1, frames),
        'blue_distance': blue_distance / max(1, frames),
    }


def benchmark_generations():
    folder = (ROOT / diretorio.value.strip()).resolve()
    # Also inspect the main checkpoints folder, where gen_iter_2000 may have been saved.
    search_folders = [folder]
    if folder != ROOT / 'checkpoints':
        search_folders.append(ROOT / 'checkpoints')
    checkpoints = []
    for search_folder in search_folders:
        checkpoints.extend(list_generation_checkpoints(search_folder, benchmark_mode.value, benchmark_stride.value))
    checkpoints = sorted({path.resolve(): path for path in checkpoints}.values(), key=lambda path: path.name)
    if not checkpoints:
        raise FileNotFoundError(f'Nenhum checkpoint gen_iter_*.pt encontrado em: {search_folders}')

    stadium_path = ROOT / 'haxball' / 'maps' / mapa.value + '.hbs'
    stadium = Stadium.load_from_file(str(stadium_path))
    stop_event = threading.Event()
    results = []
    opponents = list(BOT_FACTORIES_BENCHMARK.items())

    for checkpoint_index, checkpoint_path in enumerate(checkpoints, start=1):
        checkpoint_name = checkpoint_path.stem
        model_results = []
        for opponent_name, opponent_factory in opponents:
            wins = draws = goals_for = goals_against = kicks = touches = 0
            distances = []
            for match in range(benchmark_matches.value):
                if stop_event.is_set():
                    break
                model = RLBot(model_path=str(checkpoint_path), name=checkpoint_name)
                opponent = opponent_factory(name=f'{opponent_name}_benchmark')
                game_result = run_benchmark_match(model, opponent, stadium, benchmark_frames.value, stop_event)
                goals_for += game_result['red_score']
                goals_against += game_result['blue_score']
                wins += int(game_result['red_score'] > game_result['blue_score'])
                draws += int(game_result['red_score'] == game_result['blue_score'])
                kicks += game_result['red_kicks']
                touches += game_result['red_touches']
                distances.append(game_result['red_distance'])
            count = max(1, benchmark_matches.value)
            model_results.append({
                'checkpoint': checkpoint_name,
                'opponent': opponent_name,
                'win_rate': 100.0 * wins / count,
                'draw_rate': 100.0 * draws / count,
                'goals_for': goals_for,
                'goals_against': goals_against,
                'goal_diff': goals_for - goals_against,
                'kicks_per_match': kicks / count,
                'touches_per_match': touches / count,
                'distance_to_ball': float(np.mean(distances)) if distances else 0.0,
            })
        results.extend(model_results)
        benchmark_status.value = f'<b>Benchmark:</b> {checkpoint_index}/{len(checkpoints)} modelos avaliados'

    output_path = folder / 'benchmark_results.json'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
    return results


def summarize_benchmark(results):
    by_model = {}
    for row in results:
        by_model.setdefault(row['checkpoint'], []).append(row)
    summary = []
    for checkpoint, rows in by_model.items():
        summary.append({
            'checkpoint': checkpoint,
            'win_rate': float(np.mean([row['win_rate'] for row in rows])),
            'goal_diff': sum(row['goal_diff'] for row in rows),
            'kicks': float(np.mean([row['kicks_per_match'] for row in rows])),
            'touches': float(np.mean([row['touches_per_match'] for row in rows])),
            'distance': float(np.mean([row['distance_to_ball'] for row in rows])),
        })
    return summary


def plot_benchmark(results):
    summary = summarize_benchmark(results)
    labels = [row['checkpoint'] for row in summary]
    x = np.arange(len(labels))
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    figure.suptitle('Benchmark pós-treino: evolução das gerações', fontsize=16, fontweight='bold')
    axes[0, 0].plot(x, [row['win_rate'] for row in summary], marker='o', color='#2878c8')
    axes[0, 0].set_title('Win rate médio contra arquétipos (%)')
    axes[0, 0].set_ylim(0, 100)
    axes[0, 1].plot(x, [row['goal_diff'] for row in summary], marker='o', color='#20a464')
    axes[0, 1].set_title('Saldo de gols acumulado')
    axes[1, 0].plot(x, [row['touches'] for row in summary], marker='o', label='Toques', color='#dc6b28')
    axes[1, 0].plot(x, [row['kicks'] for row in summary], marker='o', label='Chutes', color='#824caf')
    axes[1, 0].set_title('Ações por partida')
    axes[1, 0].legend()
    axes[1, 1].plot(x, [row['distance'] for row in summary], marker='o', color='#d84a4a')
    axes[1, 1].set_title('Distância média até a bola')
    for axis in axes.flat:
        axis.set_xticks(x)
        axis.set_xticklabels(labels, rotation=45, ha='right')
        axis.grid(True, alpha=0.25)
    figure.tight_layout()
    display(figure)
    figure.savefig((ROOT / diretorio.value.strip() / 'benchmark_generations.png').resolve(), dpi=130)
    plt.close(figure)


def print_benchmark_insights(results):
    summary = summarize_benchmark(results)
    if not summary:
        return
    best = max(summary, key=lambda row: row['win_rate'])
    most_goals = max(summary, key=lambda row: row['goal_diff'])
    closest = min(summary, key=lambda row: row['distance'])
    print('\nINSIGHTS AUTOMÁTICOS')
    print(f"- Melhor win rate médio: {best['checkpoint']} ({best['win_rate']:.1f}%).")
    print(f"- Melhor saldo de gols: {most_goals['checkpoint']} ({most_goals['goal_diff']:+d}).")
    print(f"- Mais próximo da bola em média: {closest['checkpoint']} ({closest['distance']:.1f} px).")
    if len(summary) >= 2:
        first, last = summary[0], summary[-1]
        print(f"- Evolução do primeiro ao último: win rate {first['win_rate']:.1f}% -> {last['win_rate']:.1f}% ({last['win_rate'] - first['win_rate']:+.1f} pp).")
        print(f"- Toques por partida: {first['touches']:.1f} -> {last['touches']:.1f}; isso indica mais/menos envolvimento direto na jogada.")
    print('- Use o resultado por adversário no JSON para identificar especialização ou regressão contra tabeladores, pressionadores e goleiros.')


def start_benchmark(_=None):
    if benchmark_state['thread'] is not None and benchmark_state['thread'].is_alive():
        benchmark_status.value = '<b>Benchmark:</b> já está rodando'
        return
    def worker():
        try:
            results = benchmark_generations()
            with benchmark_output:
                clear_output(wait=True)
                print(f'{len(results)} confrontos agregados. Resultado salvo em benchmark_results.json.')
                plot_benchmark(results)
                print_benchmark_insights(results)
            benchmark_status.value = '<b>Benchmark:</b> concluído'
        except Exception:
            benchmark_status.value = '<b>Benchmark:</b> erro; veja a saída'
            with benchmark_output:
                import traceback
                traceback.print_exc()
    benchmark_state['thread'] = threading.Thread(target=worker, daemon=True)
    benchmark_state['thread'].start()


def stop_benchmark(_=None):
    if benchmark_state['stop'] is not None:
        benchmark_state['stop'].set()
    benchmark_status.value = '<b>Benchmark:</b> parada solicitada'

benchmark_run.on_click(start_benchmark)
benchmark_stop.on_click(stop_benchmark)

In [11]:
# Correções de ordenação, mapa e cancelamento para o benchmark

def benchmark_step(path):
    match = re.search(r'gen_iter_(\d+)', path.name)
    if match:
        return int(match.group(1))
    if 'stage1' in path.name:
        return 0
    if 'best' in path.name:
        return 10**9 - 2
    return 10**9 - 1


def benchmark_generations_fixed():
    folder = (ROOT / diretorio.value.strip()).resolve()
    search_folders = [folder]
    main_folder = ROOT / 'checkpoints'
    if main_folder != folder:
        search_folders.append(main_folder)
    checkpoints = []
    for search_folder in search_folders:
        checkpoints.extend(list_generation_checkpoints(search_folder, benchmark_mode.value, benchmark_stride.value))
    checkpoints = sorted({path.resolve(): path for path in checkpoints}.values(), key=lambda path: (benchmark_step(path), path.name))
    if not checkpoints:
        raise FileNotFoundError(f'Nenhum checkpoint gen_iter_*.pt encontrado em: {search_folders}')

    stadium_path = ROOT / 'haxball' / 'maps' / f'{mapa.value}.hbs'
    stadium = Stadium.load_from_file(str(stadium_path))
    stop_event = threading.Event()
    benchmark_state['stop'] = stop_event
    results = []
    opponents = list(BOT_FACTORIES_BENCHMARK.items())

    for checkpoint_index, checkpoint_path in enumerate(checkpoints, start=1):
        if stop_event.is_set():
            break
        checkpoint_name = checkpoint_path.stem
        for opponent_name, opponent_factory in opponents:
            wins = draws = goals_for = goals_against = kicks = touches = 0
            distances = []
            for _ in range(benchmark_matches.value):
                if stop_event.is_set():
                    break
                model = RLBot(model_path=str(checkpoint_path), name=checkpoint_name)
                opponent = opponent_factory(name=f'{opponent_name}_benchmark')
                game_result = run_benchmark_match(model, opponent, stadium, benchmark_frames.value, stop_event)
                goals_for += game_result['red_score']
                goals_against += game_result['blue_score']
                wins += int(game_result['red_score'] > game_result['blue_score'])
                draws += int(game_result['red_score'] == game_result['blue_score'])
                kicks += game_result['red_kicks']
                touches += game_result['red_touches']
                distances.append(game_result['red_distance'])
            count = max(1, benchmark_matches.value)
            results.append({
                'checkpoint': checkpoint_name,
                'opponent': opponent_name,
                'win_rate': 100.0 * wins / count,
                'draw_rate': 100.0 * draws / count,
                'goals_for': goals_for,
                'goals_against': goals_against,
                'goal_diff': goals_for - goals_against,
                'kicks_per_match': kicks / count,
                'touches_per_match': touches / count,
                'distance_to_ball': float(np.mean(distances)) if distances else 0.0,
            })
        benchmark_status.value = f'<b>Benchmark:</b> {checkpoint_index}/{len(checkpoints)} modelos avaliados'

    output_path = folder / 'benchmark_results.json'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
    return results


benchmark_generations = benchmark_generations_fixed

In [13]:
# Métrica de toque físico: o motor atual só emite eventos de chute

def run_benchmark_match(red_bot, blue_bot, stadium, frames, stop_event):
    game = HaxBallGame(stadium=stadium, score_limit=3, time_limit_secs=0)
    red_bot.reset()
    blue_bot.reset()
    red = next(player for player in game.players if player.team == Team.RED)
    blue = next(player for player in game.players if player.team == Team.BLUE)
    red_kicks = blue_kicks = red_touches = blue_touches = 0
    red_distance = blue_distance = 0.0

    for _ in range(frames):
        if stop_event.is_set():
            break
        red_contact = red.pos.distance_to(game.ball.pos) <= red.radius + game.ball.radius + 2.0
        blue_contact = blue.pos.distance_to(game.ball.pos) <= blue.radius + game.ball.radius + 2.0
        red_cmd = red_bot.act(game, red)
        blue_cmd = blue_bot.act(game, blue)
        info = game.step({red.player_id: red_cmd, blue.player_id: blue_cmd})
        red_kicks += sum(1 for event in info.get('events', {}).get('kicks', []) if event.get('player_id') == red.player_id)
        blue_kicks += sum(1 for event in info.get('events', {}).get('kicks', []) if event.get('player_id') == blue.player_id)
        red_touches += int(red_contact)
        blue_touches += int(blue_contact)
        red_distance += red.pos.distance_to(game.ball.pos)
        blue_distance += blue.pos.distance_to(game.ball.pos)

    return {
        'red_score': game.red_score,
        'blue_score': game.blue_score,
        'red_kicks': red_kicks,
        'blue_kicks': blue_kicks,
        'red_touches': red_touches,
        'blue_touches': blue_touches,
        'red_distance': red_distance / max(1, frames),
        'blue_distance': blue_distance / max(1, frames),
    }